# Ontology Generation Agent — Server-Side Batch Evaluation

This notebook evaluates the **deployed** Ontology Generation Agent (`OntologyRuntimeArn`)
**entirely server-side** using Amazon Bedrock **AgentCore Batch Evaluations**
(`bedrock_agentcore.evaluation.BatchEvaluationRunner`). It mirrors
`1_metadata_agent_ac_eval.ipynb` (the metadata-enrichment agent) but targets
`agents/ontology_agent`: the agent generates an OWL/RDF ontology (N-Quads) from the table
schemas, persists per-table fragments to S3, enriches the Glue Data Catalog, and pushes
triples to Neptune — one batch of N tables per session.

**No scoring runs locally.** The previous revision used the on-demand `Evaluation.run`
toolkit (span-by-`cloud.resource_id`, prone to *"No spans found"*) **plus a local
`bedrock_runtime.converse` ontology-quality judge** reading S3 N-Quads. This revision runs
**built-in + custom evaluators entirely in-service**: span discovery is by **service name +
session id + time range**, and the ontology-quality judge is a **server-side custom SESSION
LLM-as-Judge** that reads the agent's generated N-Quads directly from the session
`{context}` (the agent's `append_nquads` / `save_intermediate_ontology` tool calls) — no
Lambda, no local model call.

### What gets scored (all server-side)
**Built-in evaluators** (reason from the session trace + assertions)

| Evaluator | Level | Ground truth |
|:--|:--|:--|
| `Builtin.GoalSuccessRate` | Session | `assertions` |

**Custom LLM-as-Judge evaluator** (created with `create_evaluator`, scored in-service)

| Evaluator | Level | Placeholders | Checks |
|:--|:--|:--|:--|
| `OntologyQuality` | SESSION | `{context}`, `{actual_tool_trajectory}` | OWL class/property coverage, FK relationship modeling, and `rdfs:comment` annotation quality of the generated N-Quads |

### Prerequisites
- **AgentCore stack** deployed (`{PROJECT_NAME}-agentcore`); `OntologyRuntimeArn` populated
- `bedrock-agentcore>=1.11.0` and `bedrock-agentcore-starter-toolkit==0.3.9`
- AWS credentials with access to Bedrock AgentCore, DynamoDB, Glue, S3, CloudWatch Logs

In [1]:
# Prereq (uncomment if not already installed):
# !pip install 'bedrock-agentcore>=1.11.0' bedrock-agentcore-starter-toolkit==0.3.9 boto3 pandas python-dotenv --quiet

In [2]:
# Verify the AgentCore Evaluation SDK is available
try:
    import bedrock_agentcore  # noqa: F401
    from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
        BatchEvaluationRunner,  # noqa: F401
    )
    print("✓ bedrock-agentcore installed:",
          getattr(bedrock_agentcore, "__version__", "version unknown"))
    print("✓ BatchEvaluationRunner import OK")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("  Install with: pip install 'bedrock-agentcore>=1.11.0'")
    raise

✓ bedrock-agentcore installed: version unknown
✓ BatchEvaluationRunner import OK


In [3]:
import re

def _redact_account_ids(obj):
    """Recursively mask AWS account IDs in ARN strings to their last 4 digits (********NNNN)."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(\d{8})(\d{4})', r'********\2', obj)
    return obj


def _mask_acct(text):
    """Mask AWS account IDs (12-digit runs) to their last 4 digits — e.g. ********2087 —
    in any string (ARNs, bucket names, plain ids). Never emits the full account number."""
    import re as __re
    if not isinstance(text, str):
        return text
    return __re.sub(r'(\d{8})(\d{4})', r'********\2', text)


## 1. Setup & Credentials

In [4]:
import os
import sys  # noqa: F401  (kept for parity with sibling eval notebooks)
import json
import time
import uuid
import pandas as pd
import boto3
from botocore.config import Config
from datetime import datetime, timezone

from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)
from bedrock_agentcore.evaluation.runner.dataset_types import (
    Dataset, PredefinedScenario, Turn,
)
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput,
)

print("✓ Imports successful")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


✓ Imports successful


In [5]:
# Load environment variables from .env file (sourced from cdk/cdk-outputs-agentcore.json)
from dotenv import load_dotenv, set_key

ENV_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd(), '.env')
load_dotenv(dotenv_path=ENV_FILE, override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')
session = boto3.Session(profile_name=os.environ.get('AWS_PROFILE', 'default'))
region = session.region_name or 'us-east-1'

# ── CloudFormation output resolver ──
# For any env var still set to 'XXX' (no .env or .env missing the key), resolve from
# live CloudFormation stack outputs. Stacks may be deployed with a 'dev' suffix.
def _get_stack_outputs(stack_suffix: str) -> dict:
    """Return {OutputKey: OutputValue} for '{PROJECT_NAME}[-dev]-{suffix}', or {} if absent."""
    cfn = session.client('cloudformation', region_name=region)
    for name in (f'{PROJECT_NAME}-dev-{stack_suffix}', f'{PROJECT_NAME}-{stack_suffix}'):
        try:
            resp = cfn.describe_stacks(StackName=name)
            return {o['OutputKey']: o['OutputValue'] for o in resp['Stacks'][0].get('Outputs', [])}
        except Exception:
            continue
    return {}

_CFN_MAP = {
    'ARTIFACTS_BUCKET':        ('data-lake',  'ArtifactsBucketName'),
    'ATHENA_WORKGROUP':        ('athena',     'AthenaWorkgroupName'),
    'ONTOLOGY_METADATA_TABLE': ('dynamodb',   'MetadataTableName'),
    'KNOWLEDGE_BASE_ID':       ('bedrock-kb', 'OntologyPatternsKbId'),
}
_cfn_cache = {}
_resolved_from_cfn = []
for env_var, (stack, output_key) in _CFN_MAP.items():
    current = os.environ.get(env_var, 'XXX')
    if current != 'XXX':
        continue
    if stack not in _cfn_cache:
        _cfn_cache[stack] = _get_stack_outputs(stack)
    value = _cfn_cache[stack].get(output_key, 'XXX')
    os.environ[env_var] = value
    if value != 'XXX':
        _resolved_from_cfn.append(env_var)
if _resolved_from_cfn:
    for env_var in _resolved_from_cfn:
        set_key(ENV_FILE, env_var, os.environ[env_var])
    print(f"✓ Wrote {len(_resolved_from_cfn)} CFN-resolved variable(s) to .env: {', '.join(_resolved_from_cfn)}")

print("\n✓ Environment variables configured:")
for key in ['ARTIFACTS_BUCKET', 'ATHENA_WORKGROUP', 'ONTOLOGY_METADATA_TABLE', 'KNOWLEDGE_BASE_ID']:
    print(f"  {key} = {_mask_acct(str(os.environ.get(key, '(not set)')))}")


✓ Environment variables configured:
  ARTIFACTS_BUCKET = semantic-layer-dev-artifacts-********2087
  ATHENA_WORKGROUP = semantic-layer-dev-workgroup
  ONTOLOGY_METADATA_TABLE = semantic-layer-dev-metadata
  KNOWLEDGE_BASE_ID = 5HWK8RDUUR


In [6]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# Invoke the JWT-inbound runtime with a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    cfg = lam.get_function_configuration(FunctionName=f'{PROJECT_NAME}-mcp-tools')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


In [7]:
# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
account_id = identity['Account']
print(f"✓ AWS Credentials Verified:")
print(f"  Profile:   {os.environ.get('AWS_PROFILE', 'default')}")
print(f"  Account:   ...{account_id[-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region:    {region}")

✓ AWS Credentials Verified:
  Profile:   huthmac+demo-Admin
  Account:   ...4087
  User/Role: huthmac-Isengard
  Region:    us-east-1


## 2. Resolve the AgentCore Ontology Runtime

Read the deployed ontology-generation runtime ARN from the `{PROJECT_NAME}-agentcore`
CloudFormation stack and derive the agent ID from it. The agent ID is what AgentCore
Evaluations keys evaluator results on.

In [8]:
ac = _get_stack_outputs('agentcore')
ONTOLOGY_RUNTIME_ARN = ac['OntologyRuntimeArn']
agent_id = ONTOLOGY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"✓ Runtime ARN: {_mask_acct(ONTOLOGY_RUNTIME_ARN)}")
print(f"✓ Agent ID:    {agent_id}")

✓ Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:********2087:runtime/semantic_layer_dev_ontology-iddx7A6C74
✓ Agent ID:    semantic_layer_dev_ontology-iddx7A6C74


## 3. Define Test Cases — S3 Tables (Iceberg) Discovery

Discover all tables in the curated namespace and define one test case = one config = one
runtime invocation = one session of N tables. `MAX_TABLES` lets you smoke-test with 1 table;
set to 0 for the full namespace. Ontology generation is two-phase (per-table N-Quads +
holistic FK refinement), so a full-namespace run is long — start with `MAX_TABLES=1`.

In [9]:
# Resolve catalog/database coordinates from CloudFormation if still 'XXX'.
datalake = _cfn_cache.get('data-lake') or _get_stack_outputs('data-lake')
_cfn_cache['data-lake'] = datalake

S3T_DATABASE = os.environ.get('S3T_DATABASE') or datalake.get('Namespace', 'normalized')
S3T_CATALOG = os.environ.get('S3T_CATALOG')
if not S3T_CATALOG:
    prefix = datalake.get('S3TablesFederatedCatalogName', 's3tablescatalog')
    S3T_CATALOG = f"{prefix}/{PROJECT_NAME}-dev-analytics-tables"

print("✓ Catalog/database configured:")
print(f"  S3T_CATALOG  = {S3T_CATALOG}")
print(f"  S3T_DATABASE = {S3T_DATABASE}")

def s3t(table_name: str) -> dict:
    """Build the S3 Tables (Iceberg) data-source dict the agent expects in DynamoDB."""
    return {
        'databaseName': S3T_DATABASE,
        'tableName':    table_name,
        'catalogId':    S3T_CATALOG,
        'dataSource':   'AwsDataCatalog',
        'tableId':      f'{S3T_CATALOG}::{S3T_DATABASE}.{table_name}',
    }

# ── Discover all tables in the S3 Tables namespace ──
# Federated S3 Tables catalogs require CatalogId of "<account>:<S3T_CATALOG>"
glue_catalog_id = f"{account_id}:{S3T_CATALOG}"
glue = session.client('glue', region_name=region)
paginator = glue.get_paginator('get_tables')
all_table_names = []
for page in paginator.paginate(CatalogId=glue_catalog_id, DatabaseName=S3T_DATABASE):
    all_table_names.extend(t['Name'] for t in page.get('TableList', []))
all_table_names.sort()
print(f"\n✓ Discovered {len(all_table_names)} tables in {S3T_CATALOG}.{S3T_DATABASE}")

# Smoke-test slice: set MAX_TABLES=1 for a one-table run; 0 = full namespace.
MAX_TABLES = int(os.environ.get('MAX_TABLES', '1'))
if MAX_TABLES > 0:
    selected_table_names = all_table_names[:MAX_TABLES]
    print(f"  ⚠ MAX_TABLES={MAX_TABLES} — using a {len(selected_table_names)}-table subset")
else:
    selected_table_names = list(all_table_names)
    print(f"  Using all {len(selected_table_names)} tables")

if not selected_table_names:
    raise RuntimeError(f"No tables found in {S3T_CATALOG}.{S3T_DATABASE} — cannot build test case")

# ── De-layering enrichment brief (docs/plan/2026-06-26-semantic-layer-enrichments.md) ──
# Same brief nb1 (RAG) seeds. The VKG agent reads uploadedDocuments on its INITIAL build
# (ontology_agent/prompt_builder.build_phase1_table_prompt reads config['uploadedDocuments']
# directly, independent of revisionMode) and folds each §3 recipe into the rdfs:comment of
# the matching class/property IRI. Without it, the generator prompts have no
# source for the join/derivation/label recipes. See the design doc §1/§4.3.
ENRICHMENT_BRIEF_PATH = os.path.join(
    os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd(),
    '..', 'data', 'eval', 'normalized_layer_enrichment_brief.md',
)
if not os.path.exists(ENRICHMENT_BRIEF_PATH):
    raise FileNotFoundError(
        f"Enrichment brief not found at {ENRICHMENT_BRIEF_PATH} — it is the de-layering "
        f"annotation map (docs/plan/2026-06-26-semantic-layer-enrichments.md §3) and MUST "
        f"be seeded, or the generator prompts have no source for the join/"
        f"derivation/label recipes and VKG GoalSuccess regresses on gt-00/02/03/04/06/07."
    )

# §2 query-semantics note. The VKG useCasesDescription below is intentionally tuned for
# ontology STRUCTURE (FIBO/ACORD class/property shape) — that drives OntologyQuality — so
# we APPEND the §2 query-semantics guidance rather than replacing it. The §2 data-sources
# note (bridges + unprefixed-FK transforms) is appended to dataSourcesDescription.
LAYER_USE_CASES_QUERY_NOTE = (
    " Downstream natural-language answers must GROUP BY human-readable labels (not surrogate "
    "codes), join through the documented bridge tables, and reproduce any declared key "
    "transforms verbatim — capture those join/label/derivation recipes as rdfs:comment "
    "annotations on the relevant class/property IRIs (see the uploaded enrichment brief)."
)
LAYER_DATA_SOURCES_NOTE = (
    " Several relationships are bridged through junction tables and some FKs are stored in a "
    "different surface form than the referenced PK (e.g. an unprefixed id) — every such "
    "join/transform is stated in the uploaded enrichment brief and must be annotated exactly."
)

_VKG_USE_CASES = (
    'Insurance policy & coverage analytics: build an OWL ontology (virtual knowledge graph) '
    'over the curated insurance semantic layer (parties, policies, coverages, riders, payouts) '
    'so analysts can ask graph-grounded natural-language questions, with FIBO/ACORD-aligned '
    'class and property structure.'
)

test_cases = [
    {
        'id': 'ontology_curated_layer',
        'category': 'multi_s3table_ontology',
        # Path to the §3 reference brief; agent_invoker uploads it per-case (the S3
        # key is layer-id-scoped) and adds it to uploadedDocuments before invoke.
        'enrichment_brief_path': ENRICHMENT_BRIEF_PATH,
        'dynamodb_config': {
            'name': 'curated-layer-ontology',
            'type': 'VKG',
            'dataSourcesDescription': (
                f'All {len(selected_table_names)} Iceberg S3 Tables in the {S3T_DATABASE} '
                f'namespace.{LAYER_DATA_SOURCES_NOTE}'
            ),
            'useCasesDescription': _VKG_USE_CASES + LAYER_USE_CASES_QUERY_NOTE,
            'dataSources': [s3t(name) for name in selected_table_names],
        },
    },
]

print(f"\n✓ Defined {len(test_cases)} test case(s)")
print(f"  Enrichment brief: {os.path.relpath(ENRICHMENT_BRIEF_PATH)}")
for tc in test_cases:
    n = len(tc['dynamodb_config']['dataSources'])
    print(f"  {tc['id']} ({tc['category']}): {n} table(s)")

✓ Catalog/database configured:
  S3T_CATALOG  = s3tablescatalog/semantic-layer-dev-analytics-tables
  S3T_DATABASE = normalized



✓ Discovered 40 tables in s3tablescatalog/semantic-layer-dev-analytics-tables.normalized
  Using all 40 tables

✓ Defined 1 test case(s)
  Enrichment brief: ../data/eval/normalized_layer_enrichment_brief.md
  ontology_curated_layer (multi_s3table_ontology): 40 table(s)


In [10]:
# Helper to seed a semantic-layer-metadata DynamoDB record (replicates the admin UI flow).
METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE', 'semantic-layer-metadata')


def upload_enrichment_brief(*, layer_id: str, local_path: str) -> dict:
    """Upload the de-layering enrichment brief to the layer's S3 prefix.

    Mirrors the admin create-layer wizard's document-upload path
    (ontology_service.upload_metadata_file → s3://<artifacts>/ontologies/{id}/uploaded/),
    so the deployed ontology agent lists it under UPLOADED REFERENCE DOCUMENTS and reads
    it while generating each table's OWL/RDF fragment.

    Parameters:
        layer_id: the DynamoDB `id` partition key for this layer (scopes the S3 key).
        local_path: filesystem path to the brief markdown file.

    Returns:
        The uploadedDocuments entry dict: {filename, path (s3 uri), size (bytes)}.
    """
    bucket = os.environ['ARTIFACTS_BUCKET']
    filename = os.path.basename(local_path)
    key = f"ontologies/{layer_id}/uploaded/{filename}"
    session.client('s3', region_name=region).upload_file(local_path, bucket, key)
    return {
        'filename': filename,
        'path': f"s3://{bucket}/{key}",
        'size': os.path.getsize(local_path),
    }


def seed_test_ontology(ontology_id: str, tables: list, *, name: str = 'vkg-eval-test',
                       ontology_type: str = 'VKG', data_sources_description: str = '',
                       use_cases_description: str = '',
                       created_by: str = 'eval@semantic-layer.local',
                       uploaded_documents: list | None = None) -> None:
    """Write a DynamoDB v1 record matching the shape created by the admin UI flow.

    Parameters:
        ontology_id: the `id` partition key the agent will read.
        tables: list of full data-source dicts (from `s3t()`).
        name: human-readable config name; gets a UTC timestamp suffix.
        ontology_type: 'VKG' for the ontology (virtual knowledge graph) agent.
        data_sources_description: optional free-text description of the sources.
        use_cases_description: optional free-text description of the intended use cases.
        created_by: user/email recorded as creator.
        uploaded_documents: optional list of {filename, path, size} reference-document
            dicts (from `upload_enrichment_brief`). Persisted as `uploadedDocuments`,
            which the ontology agent reads on the INITIAL Phase-1 build
            (prompt_builder.build_phase1_table_prompt) — fires on first build.
    """
    ddb_table = session.resource('dynamodb').Table(METADATA_TABLE)
    now = datetime.now(timezone.utc).isoformat()
    ts = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
    name_with_ts = f"{name}-{ts}"
    data_sources = [
        {
            'databaseName': t['databaseName'], 'tableName': t['tableName'],
            'catalogId': t.get('catalogId', 'AWSDataCatalog'),
            'dataSource': t.get('dataSource', 'AwsDataCatalog'),
            'tableId': t.get('tableId', f"{t.get('catalogId', 'AWSDataCatalog')}::{t['databaseName']}.{t['tableName']}"),
        }
        for t in tables
    ]
    item = {
        'id': ontology_id, 'version': 'v1', 'type': ontology_type, 'name': name_with_ts,
        'status': 'pending', 'configuration': {}, 'dataSources': data_sources,
        'createdAt': now, 'updatedAt': now, 'buildStartedAt': now, 'createdBy': created_by,
    }
    if data_sources_description:
        item['dataSourcesDescription'] = data_sources_description
    if use_cases_description:
        item['useCasesDescription'] = use_cases_description
    if uploaded_documents:
        item['uploadedDocuments'] = uploaded_documents
    ddb_table.put_item(Item=item)
    docs_note = f", {len(uploaded_documents)} ref-doc(s)" if uploaded_documents else ""
    print(f"✓ Seeded DynamoDB item: {ontology_id} → name='{name_with_ts}' ({len(data_sources)} table(s){docs_note})")

print("✓ seed_test_ontology + upload_enrichment_brief defined")

✓ seed_test_ontology + upload_enrichment_brief defined


## 4. Create the Custom Ontology-Quality Evaluator — server-side, no Lambda

This SESSION LLM-as-Judge runs **inside AgentCore Evaluations**. The `{context}` placeholder
carries the full session conversation — including the ontology agent's `append_nquads` and
`save_intermediate_ontology` tool calls (the generated OWL/RDF N-Quads) and its
`get_single_table_schema` tool responses (the source columns). The judge reads both directly,
so a local `bedrock_runtime.converse` judge that fetched N-Quads markdown from S3 is no
longer needed.

> Re-running this cell creates a **new** evaluator (unique name suffix).

In [11]:
_cp = session.client('bedrock-agentcore-control', region_name=region)
_SUFFIX = uuid.uuid4().hex[:8]
JUDGE_MODEL_ID = 'global.anthropic.claude-sonnet-5'

# A 0.0–1.0 quality scale (continuous), unlike the binary query-agent judges: ontology
# quality is a matter of degree (coverage / modeling / annotation completeness).
_QUALITY_SCALE = {
    'numerical': [
        {'value': 0.0, 'label': 'poor',
         'definition': 'No owl:Class, almost no properties, or no annotations.'},
        {'value': 0.5, 'label': 'partial',
         'definition': 'Class present but many columns unmodeled, some FKs missed, or shallow comments.'},
        {'value': 1.0, 'label': 'good',
         'definition': 'Full class/property coverage, FK ObjectProperties modeled, business-meaningful rdfs:comments.'},
    ]
}

def _create_llm_judge(*, name: str, level: str, instructions: str) -> str:
    """Create an LLM-as-Judge evaluator and return its evaluatorId.

    Parameters:
        name: human-readable evaluator name (a unique suffix is appended).
        level: 'TRACE' or 'SESSION' — the granularity the service scores at.
        instructions: judge prompt; may reference {context}/{actual_tool_trajectory} etc.

    Returns:
        The created evaluator's evaluatorId.
    """
    resp = _cp.create_evaluator(
        evaluatorName=f"{name}_{_SUFFIX}",
        level=level,
        evaluatorConfig={
            'llmAsAJudge': {
                'instructions': instructions,
                'ratingScale': _QUALITY_SCALE,
                'modelConfig': {
                    'bedrockEvaluatorModelConfig': {
                        'modelId': JUDGE_MODEL_ID,
                        'inferenceConfig': {'maxTokens': 4096},
                    }
                },
            }
        },
    )
    return resp['evaluatorId']


ONTOLOGY_QUALITY_ID = _create_llm_judge(
    name='OntologyQuality',
    level='SESSION',
    instructions=(
        "You are an expert OWL/RDF ontology evaluator scoring a deployed ontology-generation "
        "agent's output for one session covering one or more tables.\n\n"
        "Session context (full conversation, including tool calls and tool results):\n"
        "{context}\n\n"
        "Available tools: {available_tools}\n"
        "Actual tool trajectory: {actual_tool_trajectory}\n\n"
        "In the context, locate:\n"
        "  (a) the OUTPUT of `get_single_table_schema` — the source table columns; and\n"
        "  (b) the ARGUMENTS of `append_nquads` / `save_intermediate_ontology` — the OWL/RDF "
        "N-Quads the agent generated.\n\n"
        "Score the generated ontology on three axes, then return the MEAN as your score:\n"
        "  1. class/property coverage — exactly one owl:Class per table AND an "
        "owl:DatatypeProperty for (nearly) every source column, each with rdfs:domain/rdfs:range.\n"
        "  2. relationship modeling — foreign-key-like columns (e.g. *_id referencing another "
        "table) modeled as owl:ObjectProperty with rdfs:domain/rdfs:range to the related class "
        "(or the table genuinely has no FKs and none are invented).\n"
        "  3. annotation quality — the owl:Class and most properties carry a business-meaningful "
        "rdfs:comment / rdfs:label (not just the column name restated).\n\n"
        "Score 1.0 (good) when all three axes are strong; 0.5 (partial) when coverage is "
        "incomplete, some FKs are missed, or comments are shallow; 0.0 (poor) when there is no "
        "owl:Class, almost no properties, or no annotations. If no N-Quads are present in the "
        "context, score 0.0 and say the generated ontology was not found. Briefly justify the score."
    ),
)

CUSTOM_EVALUATOR_IDS = [ONTOLOGY_QUALITY_ID]
print("✓ Custom server-side evaluator created (LLM-as-Judge, no Lambda):")
print(f"  OntologyQuality (SESSION) : {ONTOLOGY_QUALITY_ID}")

✓ Custom server-side evaluator created (LLM-as-Judge, no Lambda):
  OntologyQuality (SESSION) : OntologyQuality_981b192b-uExXolALwc


## 5. Build the Dataset + agent_invoker

`BatchEvaluationRunner.run_dataset_evaluation(config, dataset, agent_invoker)` invokes the
agent **once per scenario** via our `agent_invoker`, waits for CloudWatch ingestion, then
submits a single batch-evaluation job and polls until completion.

**The ontology agent is asynchronous** (background thread + DynamoDB progress), so the
invoker must: (1) seed a fresh DynamoDB config, (2) `invoke_agent_runtime` with the
framework-managed `session_id` as `runtimeSessionId` (so the batch service can correlate
spans), (3) **block** by polling DynamoDB until status is `completed`/`failed`, and (4)
return the agent's summary as `agent_output`.

In [12]:
agentcore_client = session.client('bedrock-agentcore', region_name=region)
ddb_resource = session.resource('dynamodb')
ddb_table = ddb_resource.Table(METADATA_TABLE)

# Map each scenario_id -> the seeded DynamoDB config id + table list.
SCENARIO_RUNS: dict = {}
_TC_BY_SCENARIO = {tc['id']: tc for tc in test_cases}


def _server_side_seconds(*, item: dict):
    """Compute build latency from the DynamoDB record's own timestamps.

    The agent stamps `buildStartedAt` on the processing transition and `completedAt`
    on completion. Returns None if either timestamp is missing/unparseable (we never
    fabricate a value).

    Parameters:
        item: the final DynamoDB item for the ontology build job.
    Returns:
        elapsed seconds as float, or None if timestamps are unavailable.
    """
    started_raw = item.get('buildStartedAt')
    completed_raw = item.get('completedAt')
    if not started_raw or not completed_raw:
        return None
    try:
        return (datetime.fromisoformat(completed_raw) - datetime.fromisoformat(started_raw)).total_seconds()
    except (TypeError, ValueError):
        return None


def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Drive the deployed ontology-generation agent for one scenario and block until the
    async build finishes (so its spans exist for batch discovery).

    BatchEvaluationRunner supplies invoker_input.payload (the Turn.input = the scenario id)
    and invoker_input.session_id (framework-managed runtimeSessionId).
    """
    scenario_id = (invoker_input.payload if isinstance(invoker_input.payload, str)
                   else json.dumps(invoker_input.payload))
    tc = _TC_BY_SCENARIO[scenario_id]
    cfg = tc['dynamodb_config']
    tables = cfg['dataSources']
    num_tables = len(tables)

    case_id = f"vkg-{tc['id']}-{uuid.uuid4().hex[:8]}"
    print(f"\n=== scenario '{scenario_id}' -> case_id={case_id} ({num_tables} tables) ===")
    print(f"  runtimeSessionId = {invoker_input.session_id}")

    # Upload the de-layering enrichment brief to THIS layer's S3 prefix (the key is
    # layer-id-scoped, matching the admin flow) and attach it as a reference doc.
    # The VKG agent reads it on the INITIAL Phase-1 build and folds each §3 recipe into
    # the rdfs:comment of the matching class/property IRI. See the design doc §1/§4.3.
    uploaded_documents = None
    brief_path = tc.get('enrichment_brief_path')
    if brief_path:
        brief = upload_enrichment_brief(layer_id=case_id, local_path=brief_path)
        uploaded_documents = [brief]
        print(f"  uploaded reference brief -> {_mask_acct(brief['path'])} ({brief['size']} bytes)")

    seed_test_ontology(
        case_id, tables, name=cfg['name'], ontology_type=cfg.get('type', 'VKG'),
        data_sources_description=cfg.get('dataSourcesDescription', ''),
        use_cases_description=cfg.get('useCasesDescription', ''),
        uploaded_documents=uploaded_documents,
    )

    _invoke_runtime(
        ONTOLOGY_RUNTIME_ARN,
        invoker_input.session_id,
        json.dumps({'id': case_id}).encode('utf-8'),
    )

    # Block until the background thread reports completion. Two-phase ontology generation
    # is slower than metadata enrichment; budget 150s/table with a 10-min floor.
    max_wait_secs = max(600, 150 * num_tables)
    start = datetime.now()
    final_item, final_status = {}, 'processing'
    print(f"  Polling DynamoDB for completion (max {max_wait_secs}s)...")
    while (datetime.now() - start).total_seconds() < max_wait_secs:
        time.sleep(30)
        final_item = ddb_table.get_item(Key={'id': case_id, 'version': 'v1'}).get('Item', {})
        final_status = final_item.get('status', 'unknown')
        phase = final_item.get('phase', '')
        pct = final_item.get('progressPercent', '')
        suffix = f" — phase={phase} {pct}%" if phase else ""
        print(f"    [{(datetime.now()-start).total_seconds():.0f}s] status={final_status}{suffix}")
        if final_status in ('completed', 'failed'):
            break

    summary = final_item.get('summary') or final_item.get('error', '') or f"status={final_status}"
    build_seconds = _server_side_seconds(item=final_item)

    SCENARIO_RUNS[scenario_id] = {
        'case_id': case_id,
        'session_id': invoker_input.session_id,
        'status': final_status,
        'num_tables': num_tables,
        'tables': [t['tableName'] for t in tables],
        'summary': summary,
        'build_seconds': round(build_seconds, 1) if build_seconds is not None else None,
    }
    # Stash the most recent ontology id so downstream notebooks can reuse it.
    global ontology_id_stored
    ontology_id_stored = case_id
    lat = f"{build_seconds:.1f}s" if build_seconds is not None else 'n/a'
    print(f"  {'✓' if final_status == 'completed' else '✗'} {final_status} — {summary}")
    print(f"     latency(server)={lat}")
    return AgentInvokerOutput(agent_output=summary)


# One scenario per test case. assertions / expected_trajectory are the ground truth the
# SESSION/span built-in evaluators score against.
dataset = Dataset(
    scenarios=[
        PredefinedScenario(
            scenario_id=tc['id'],
            turns=[Turn(input=tc['id'])],  # payload routes the invoker to this tc
            expected_trajectory=[
                'get_single_table_schema',
                'append_nquads',
                'save_intermediate_ontology',
                'update_progress',
            ],
            assertions=[
                "The agent generated an OWL/RDF ontology (owl:Class + owl:DatatypeProperty with rdfs:comment) for every input table.",
                "Foreign-key relationships were modeled as owl:ObjectProperty where applicable.",
                "Per-table N-Quad fragments were persisted via save_intermediate_ontology.",
                "The final DynamoDB status is 'completed'.",
            ],
        )
        for tc in test_cases
    ]
)
print(f"✓ Dataset built: {len(dataset.scenarios)} scenario(s)")
for s in dataset.scenarios:
    print(f"  {s.scenario_id}: {len(s.assertions)} assertion(s), {len(s.expected_trajectory)} expected tool(s)")

✓ Dataset built: 1 scenario(s)
  ontology_curated_layer: 4 assertion(s), 4 expected tool(s)


## 6. Run the Batch Evaluation (server-side, two-phase + resumable)

Mirrors notebook 1's resumable split (ported here because ontology generation is even
slower and most exposed to a poll/kernel timeout):

- **Phase 1** invokes the agent once per scenario (via `agent_invoker`, which blocks on the
  async build) and **persists** the completed session id + time window + ground truth to
  `ontology_gen_sessions_latest.json` **before** any batch is submitted.
- **Phase 2** waits `ingestion_delay_seconds` for CloudWatch span ingestion, submits one
  `StartBatchEvaluation` job over those sessions, and polls to completion. It is **safe to
  re-run on its own** — it reloads the sessions from disk if they aren't in memory, so a
  Ctrl-C, poll timeout, or `FAILED` status never forces a re-invocation of the long build.

All scoring happens in-service; span discovery is by **service name + session id + time
range**.

In [13]:
import uuid as _uuid
from datetime import datetime, timezone

SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")

ALL_EVALUATOR_IDS = [
    'Builtin.GoalSuccessRate',        # SESSION — assertions
    *CUSTOM_EVALUATOR_IDS,            # custom SESSION ontology-quality judge
]

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    ingestion_delay_seconds=180,  # ontology spans land seconds before the DDB flip
)

# Recovery file: Phase 1 writes the completed session(s) here; Phase 2 reads them.
# Survives a kernel restart, so the long two-phase ontology build never has to be
# re-run just to (re)submit or re-poll the batch. Mirrors notebook 1's Phase 1/2
# split — ported here because ontology generation is even slower (per-table N-Quads
# + holistic FK refinement) and most exposed to a poll/kernel timeout.
SESSIONS_FILE = "../data/eval/results/ontology_gen_sessions_latest.json"
os.makedirs("../data/eval/results", exist_ok=True)

batch_runner = BatchEvaluationRunner(region=region)

# ── Phase 1: run each scenario's invocation ourselves, then PERSIST ──────────
# Inline (not via run_dataset_evaluation) so the completed build is captured and
# saved to disk BEFORE any batch submission can fail. agent_invoker populates
# SCENARIO_RUNS as a side effect (status / latency / case_id), exactly as the
# results cell expects.
collected_sessions = []   # sessionId / testScenarioId / window / groundTruth
for scenario in dataset.scenarios:
    sid = f"{scenario.scenario_id}-{_uuid.uuid4()}"
    start_time = datetime.now(timezone.utc)
    print(f"\n▶ Phase 1: scenario '{scenario.scenario_id}' (session_id={sid})")
    for turn in scenario.turns:
        agent_invoker(AgentInvokerInput(payload=turn.input, session_id=sid))
    end_time = datetime.now(timezone.utc)
    # Reuse the SDK's own ground-truth transform so assertions / expected_trajectory
    # are encoded identically to run_dataset_evaluation (no drift).
    ground_truth = batch_runner._transform_ground_truth(scenario)
    collected_sessions.append({
        "sessionId": sid,
        "testScenarioId": scenario.scenario_id,
        "startTime": start_time.isoformat(),
        "endTime": end_time.isoformat(),
        "groundTruth": ground_truth,
    })
    print(f"  ✓ session captured: window=[{start_time.isoformat()}, {end_time.isoformat()}]")

with open(SESSIONS_FILE, "w") as f:
    json.dump(
        {
            "agent_id": agent_id,
            "runtime_arn": ONTOLOGY_RUNTIME_ARN,
            "service_name": SERVICE_NAME,
            "log_groups": [SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
            "evaluator_ids": ALL_EVALUATOR_IDS,
            "sessions": collected_sessions,
            "scenario_runs": SCENARIO_RUNS,
        },
        f, indent=2, default=str,
    )
print(f"\n✓ Phase 1 complete — {len(collected_sessions)} session(s) persisted to {SESSIONS_FILE}")
print("  The ontology build is now saved. Phase 2 (next cell) can submit/re-submit the")
print("  batch cheaply — even after a kernel restart — without re-invoking the agent.")

SERVICE_NAME : semantic_layer_dev_ontology-iddx7A6C74.DEFAULT
LOG GROUPS   : aws/spans, /aws/bedrock-agentcore/runtimes/semantic_layer_dev_ontology-iddx7A6C74-DEFAULT



▶ Phase 1: scenario 'ontology_curated_layer' (session_id=ontology_curated_layer-dac12838-2b9b-4c99-b7eb-61f16fd5e8b5)

=== scenario 'ontology_curated_layer' -> case_id=vkg-ontology_curated_layer-204b1478 (40 tables) ===
  runtimeSessionId = ontology_curated_layer-dac12838-2b9b-4c99-b7eb-61f16fd5e8b5


  uploaded reference brief -> s3://semantic-layer-dev-artifacts-********2087/ontologies/vkg-ontology_curated_layer-204b1478/uploaded/normalized_layer_enrichment_brief.md (5503 bytes)
✓ Seeded DynamoDB item: vkg-ontology_curated_layer-204b1478 → name='curated-layer-ontology-20260627-204809' (40 table(s), 1 ref-doc(s))


  Polling DynamoDB for completion (max 6000s)...


    [30s] status=processing — phase=incremental %


...

    [5892s] status=processing — phase=refinement 95%


    [5922s] status=processing — phase=refinement 95%


    [5952s] status=processing — phase=refinement 95%


    [5982s] status=processing — phase=refinement 95%


    [6012s] status=processing — phase=refinement 95%
  ✗ processing — status=processing
     latency(server)=n/a
  ✓ session captured: window=[2026-06-27T20:48:09.312797+00:00, 2026-06-27T22:28:23.805848+00:00]

✓ Phase 1 complete — 1 session(s) persisted to ../data/eval/results/ontology_gen_sessions_latest.json
  The ontology build is now saved. Phase 2 (next cell) can submit/re-submit the
  batch cheaply — even after a kernel restart — without re-invoking the agent.


In [14]:
# ── Phase 2 — Submit + poll the batch (idempotent, re-runnable) ──────────────
# Submits ONE StartBatchEvaluation job over the session(s) captured in Phase 1 and
# polls to completion. SAFE TO RE-RUN ON ITS OWN — it reloads the sessions from
# SESSIONS_FILE if they aren't in memory (e.g. after a kernel restart), so a
# Ctrl-C, poll timeout, or FAILED status never forces a re-invocation of the long
# ontology build. The submit + poll + result-assembly mirror
# run_dataset_evaluation's second half exactly (same pre_evaluation_run_hook span
# wait, same start_batch_evaluation args, same _poll_for_results loop, same
# BatchEvaluationResult shape) — but read sessions from disk, not a live invoke.
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationResult,
    BatchEvaluationSummary,
    CloudWatchOutputDataConfig,
)

POLLING_TIMEOUT_SECONDS = 5400
POLLING_INTERVAL_SECONDS = 30

# Re-runnable: prefer in-memory sessions from Phase 1, else reload from disk.
if "collected_sessions" not in dir() or not collected_sessions:
    with open(SESSIONS_FILE) as f:
        _saved = json.load(f)
    collected_sessions = _saved["sessions"]
    agent_id = _saved["agent_id"]
    ONTOLOGY_RUNTIME_ARN = _saved["runtime_arn"]
    SERVICE_NAME = _saved["service_name"]
    SPANS_LOG_GROUP, RUNTIME_LOG_GROUP = _saved["log_groups"]
    ALL_EVALUATOR_IDS = _saved["evaluator_ids"]
    SCENARIO_RUNS = _saved.get("scenario_runs", {})
    print(f"✓ Reloaded {len(collected_sessions)} session(s) from {SESSIONS_FILE} (Phase 1 not in memory)")

if "batch_runner" not in dir():
    batch_runner = BatchEvaluationRunner(region=region)
if "batch_data_source" not in dir():
    batch_data_source = CloudWatchDataSourceConfig(
        service_names=[SERVICE_NAME],
        log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
        ingestion_delay_seconds=180,
    )

# Parse the persisted session windows back into datetimes.
_session_ids = [s["sessionId"] for s in collected_sessions]
_starts = [datetime.fromisoformat(s["startTime"]) for s in collected_sessions]
_ends = [datetime.fromisoformat(s["endTime"]) for s in collected_sessions]
_session_metadata = [
    {
        "sessionId": s["sessionId"],
        "testScenarioId": s["testScenarioId"],
        **({"groundTruth": {"inline": s["groundTruth"]}} if s.get("groundTruth") else {}),
    }
    for s in collected_sessions
]

batch_name = f"ontology_gen_batch_{uuid.uuid4().hex[:8]}"
print(f"Batch name : {batch_name}")
print(f"Evaluators : {ALL_EVALUATOR_IDS}")
print(f"Sessions   : {len(_session_ids)}")
print("Waiting for span ingestion, then submitting StartBatchEvaluation...\n")

# 1) Wait for CloudWatch span ingestion (same hook run_dataset_evaluation calls).
batch_data_source.pre_evaluation_run_hook()

# 2) Submit the batch against the EXISTING sessions (no agent invocation here).
_start_response = batch_runner.data_plane_client.start_batch_evaluation(
    batchEvaluationName=batch_name,
    evaluators=[{"evaluatorId": eid} for eid in ALL_EVALUATOR_IDS],
    dataSourceConfig=batch_data_source.to_data_source_config(
        _session_ids, min(_starts), max(_ends),
    ),
    evaluationMetadata={"sessionMetadata": _session_metadata},
    description="Server-side built-in + custom ontology-quality eval of the ontology generation agent.",
)
_batch_id = _start_response["batchEvaluationId"]
_batch_arn = _start_response["batchEvaluationArn"]
print(f"✓ Started batch: {_batch_id}")

# 3) Poll to a terminal state via the SDK's own poll loop.
_response = batch_runner._poll_for_results(
    _batch_id, POLLING_TIMEOUT_SECONDS, POLLING_INTERVAL_SECONDS,
)

# 4) Assemble a BatchEvaluationResult identically to run_dataset_evaluation — a
#    FAILED terminal status still binds batch_result with whatever results exist.
_eval_results = (
    BatchEvaluationSummary.model_validate(_response["evaluationResults"])
    if "evaluationResults" in _response else None
)
_output_data_config = None
if "outputConfig" in _response:
    _odc = _response["outputConfig"].get("cloudWatchConfig")
    if _odc:
        _output_data_config = CloudWatchOutputDataConfig(
            log_group_name=_odc["logGroupName"],
            log_stream_name=_odc["logStreamName"],
        )

batch_result = BatchEvaluationResult(
    batch_evaluation_id=_batch_id,
    batch_evaluation_arn=_batch_arn,
    batch_evaluation_name=_response["batchEvaluationName"],
    status=_response["status"],
    created_at=_response["createdAt"],
    description=_response.get("description"),
    evaluation_results=_eval_results,
    error_details=_response.get("errorDetails"),
    output_data_config=_output_data_config,
)
print(f"\n✓ Batch status: {batch_result.status}")
if batch_result.error_details:
    print(f"⚠ Error details: {batch_result.error_details}")

Batch name : ontology_gen_batch_c4d1a7c2
Evaluators : ['Builtin.GoalSuccessRate', 'OntologyQuality_981b192b-uExXolALwc']
Sessions   : 1
Waiting for span ingestion, then submitting StartBatchEvaluation...



✓ Started batch: ontology_gen_batch_c4d1a7c2-8ab6b72941


Batch evaluation ontology_gen_batch_c4d1a7c2-8ab6b72941 reached non-successful terminal status FAILED



✓ Batch status: FAILED
⚠ Error details: ['All 1 sessions failed during batch evaluation.']


## 7. Results — aggregate scores, per-scenario detail

1. **Aggregate per-evaluator scores** — from `batch_result.evaluation_results` (in-service).
2. **Per-scenario evaluator scores** — `fetch_evaluation_events()`, joined back to each
   scenario by session id.
3. **Per-scenario build metadata** — from `SCENARIO_RUNS` (status + server-side latency).

In [15]:
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f"Batch ID : {batch_result.batch_evaluation_id}")
print(f"ARN      : {_mask_acct(batch_result.batch_evaluation_arn)}")
print(f"Status   : {batch_result.status}")

# ── 1. Aggregate per-evaluator scores ──
agg_rows = []
ev = batch_result.evaluation_results
if ev is not None:
    print(f"\nSessions: completed={ev.number_of_sessions_completed} "
          f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
    for es in (ev.evaluator_summaries or []):
        stats = es.statistics
        avg = (f"{stats.average_score:.3f}"
               if stats and stats.average_score is not None else None)
        agg_rows.append({
            'Evaluator': es.evaluator_id or 'unknown',
            'AvgScore': avg,
            'Evaluated': es.total_evaluated or 0,
            'Failed': es.total_failed or 0,
        })
else:
    print("\n⚠ No aggregate evaluation_results — check job status / spans.")
df_agg = pd.DataFrame(agg_rows)

_EVAL_NAME = {ONTOLOGY_QUALITY_ID: 'OntologyQuality'}

# Map framework session id -> scenario for the per-scenario join.
_SCENARIO_BY_SESSION = {r['session_id']: sid for sid, r in SCENARIO_RUNS.items()}

# ── 2. Per-scenario evaluator events (retry: the output stream lags COMPLETED) ──
events = []
for _attempt in range(6):
    try:
        events = batch_runner.fetch_evaluation_events(batch_result)
        if events:
            break
    except (LookupError, ValueError) as exc:
        print(f"  per-scenario events not ready yet ({exc}); retrying in 20s...")
    time.sleep(20)

def _event_field(e: dict, key: str):
    """Read an event field from the top level or the nested 'attributes' map."""
    if key in e:
        return e[key]
    return (e.get('attributes', {}) or {}).get(key)

event_rows = []
for e in events:
    sess = _event_field(e, 'session.id') or _event_field(e, 'gen_ai.session.id')
    name = _event_field(e, 'gen_ai.evaluation.name')
    event_rows.append({
        'scenario_id': _SCENARIO_BY_SESSION.get(sess, sess),
        'evaluator': _EVAL_NAME.get(name, name),
        'score': _event_field(e, 'gen_ai.evaluation.score.value'),
        'label': _event_field(e, 'gen_ai.evaluation.score.label'),
        'explanation': (str(_event_field(e, 'gen_ai.evaluation.explanation') or '')),
    })
df_events = pd.DataFrame(event_rows)
print(f"\nPer-scenario evaluator events: {len(df_events)}")

# ── 3. Per-scenario build metadata ──
df_runs = pd.DataFrame(list(SCENARIO_RUNS.values()))

_lat = [r['build_seconds'] for r in SCENARIO_RUNS.values() if r.get('build_seconds') is not None]
perf = {
    'scenarios': len(SCENARIO_RUNS),
    'avg_build_seconds': round(sum(_lat) / len(_lat), 1) if _lat else None,
    'total_build_seconds': round(sum(_lat), 1) if _lat else None,
}
print("\n── Build latency (the agent doing the work) ──")
print(f"  Avg server-side latency : {perf['avg_build_seconds']}s  (n={len(_lat)})")

# ── Persist everything ──
combined_file = f"../data/eval/results/ontology_gen_batch_eval_{timestamp}.json"
combined = {
    'agent_id': agent_id,
    'runtime_arn': ONTOLOGY_RUNTIME_ARN,
    'batch_evaluation_id': batch_result.batch_evaluation_id,
    'batch_evaluation_arn': batch_result.batch_evaluation_arn,
    'status': str(batch_result.status),
    'evaluator_ids': ALL_EVALUATOR_IDS,
    'custom_evaluators': {'OntologyQuality': ONTOLOGY_QUALITY_ID},
    'aggregate_summaries': agg_rows,
    'per_scenario_events': event_rows,
    'scenario_runs': SCENARIO_RUNS,
    'performance': perf,
}
combined = _redact_account_ids(combined)
with open(combined_file, 'w') as f:
    json.dump(combined, f, indent=2, default=str)
print(f"\n✓ Wrote {combined_file}")

print("\n=== Aggregate per-evaluator scores ===")
display(df_agg)
print("=== Per-scenario evaluator scores ===")
display(df_events)
print("=== Per-scenario build metadata ===")
display(df_runs)

Batch ID : ontology_gen_batch_c4d1a7c2-8ab6b72941
ARN      : arn:aws:bedrock-agentcore:us-east-1:********2087:batch-evaluate/ontology_gen_batch_c4d1a7c2-8ab6b72941
Status   : FAILED

Sessions: completed=0 failed=1 total=1



Per-scenario evaluator events: 2

── Build latency (the agent doing the work) ──
  Avg server-side latency : Nones  (n=0)

✓ Wrote ../data/eval/results/ontology_gen_batch_eval_20260627_183255.json

=== Aggregate per-evaluator scores ===


,Evaluator,AvgScore,Evaluated,Failed
0,Builtin.GoalSuccessRate,None,0,1
1,OntologyQuality_981b192b-uExXolALwc,None,0,1


=== Per-scenario evaluator scores ===


,scenario_id,evaluator,score,label,explanation
0,ontology_curated_layer,Builtin.GoalSuccessRate,None,None,
1,ontology_curated_layer,OntologyQuality_981b192b,None,None,


=== Per-scenario build metadata ===


,case_id,session_id,status,num_tables,tables,summary,build_seconds
0,vkg-ontology_curated_layer-204b1478,ontology_curated_layer-dac12838-2b9b-4c99-b7eb-61f16fd5e8b5,processing,40,"[address, admin_codes, annuity_detail, arrangement_destination, arrangement_source, carrier_appointment, coverage, coverage_product, distribution_level, email_address, financial_activity, financial_statement, govt_id_info, holding, holding_activity, holding_arrangement, holding_dbg, holding_loan, holding_payout, holding_projection, holding_restriction, holding_subaccount, invest_product, life_detail, life_participant, loan_activity, party, party_banking, party_license, phone, policy_loan_summary, policy_product, producer_agreement, reinsurance_info, relation, rider, rider_participant, subaccount_activity, substandard_rating, type_codes]",status=processing,None


## 8. Store IDs for Downstream Notebooks

In [16]:
ontology_gen_batch_id = batch_result.batch_evaluation_id
%store ontology_id_stored
%store ontology_gen_batch_id

print("✓ Stored for downstream notebooks:")
print(f"  ontology_id_stored    = {ontology_id_stored}")
print(f"  ontology_gen_batch_id = {ontology_gen_batch_id}")
print(f"\n  Combined results: {combined_file}")

Stored 'ontology_id_stored' (str)
Stored 'ontology_gen_batch_id' (str)
✓ Stored for downstream notebooks:
  ontology_id_stored    = vkg-ontology_curated_layer-204b1478
  ontology_gen_batch_id = ontology_gen_batch_c4d1a7c2-8ab6b72941

  Combined results: ../data/eval/results/ontology_gen_batch_eval_20260627_183255.json


## Summary

This notebook evaluates the **deployed** ontology generation agent **entirely server-side**
via AgentCore Batch Evaluations — no local scoring.

### Pipeline
1. **Custom evaluator** (LLM-as-Judge, no Lambda) — `OntologyQuality` (SESSION), created with
   `create_evaluator` and scored in-service from the session `{context}` (the agent's
   generated N-Quads + source schema).
2. **Dataset** — one `PredefinedScenario` per test case, with `assertions` + an
   `expected_trajectory` (the ground truth the built-in evaluators score against).
3. **agent_invoker** — for each scenario: seed a DynamoDB config, `invoke_agent_runtime`
   with the framework session id, and **block** on DynamoDB until the build completes so the
   agent's CloudWatch spans exist. Records server-side latency (`completedAt − buildStartedAt`).
4. **Two-phase, resumable batch** (mirrors notebook 1): **Phase 1** invokes + persists the
   completed sessions to `ontology_gen_sessions_latest.json`; **Phase 2** submits one
   `StartBatchEvaluation` job over those sessions and polls to completion. Phase 2 is safe to
   re-run on its own (reloads sessions from disk), so a poll timeout / kernel restart never
   re-invokes the long build.
5. **Results** — aggregate per-evaluator scores, per-scenario scores via
   `fetch_evaluation_events`, and per-scenario build metadata.

### Evaluators (all server-side)
- `Builtin.GoalSuccessRate` — build goal achieved (uses the scenario assertions)
- `OntologyQuality` (custom SESSION judge) — class/property coverage, FK relationship
  modeling, and `rdfs:comment` annotation quality, scored from the session context

> The AWS-managed `ToolSelectionAccuracy` / `ToolParameterAccuracy` builtins are intentionally
> NOT used: like the query agents, this agent's deterministic phases make the managed
> tool-trajectory judges dock spans they shouldn't (ReAct-trajectory assumption). `OntologyQuality`
> + `GoalSuccessRate` are the signal.

### Tuning knobs
- `MAX_TABLES` env var — slice the namespace for smoke tests (default 1; 0 = full namespace)
- `ingestion_delay_seconds` (180) — wait for CloudWatch span ingestion before evaluating
- `POLLING_TIMEOUT_SECONDS` (5400) — covers full-namespace generation + the batch job